# Pure PP-OCRv5 light preparation (Kaggle, single run)

Produces the `pdf-inspector -> PP-OCRv5` light prep with no Tesseract in the path.
Everything happens in this notebook: it clones the repo, installs `pdf-inspector` from PyPI,
runs the inspector over the corpus, and OCRs every page the inspector flags.

**One-time setup**
1. Upload `0.1. BENCHMARK/` as a Kaggle Dataset (266 MB). Keep the folder name.
   It is reusable for any later GPU work on this bundle.
2. Attach the dataset to this notebook.
3. Settings: accelerator **GPU T4**, **Internet on**.
4. Private repo: add a GitHub token as the Kaggle Secret `GITHUB_TOKEN`.

Then run top to bottom and download `light_ocr_ppocrv5.zip`. Roughly 15 minutes:
about 1 minute of inspector and about 13 minutes of OCR at the measured 0.87 s/page on a T4.
The OCR loop resumes by unit, so rerunning the cell after a crash only redoes what is missing.

In [ ]:
import glob
import os
import subprocess

REPO = "https://github.com/iSE-UET-VNU/AXIOM_DE-RD"
BRANCH = "feature/retrieval-baseline"
CHECKOUT = "/kaggle/working/AXIOM_DE-RD"

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://{token}@") if token else REPO

if not os.path.exists(CHECKOUT):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CHECKOUT], check=True)
print(subprocess.run(["git", "-C", CHECKOUT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
bench = next((p for p in glob.glob("/kaggle/input/*/0.1. BENCHMARK") if os.path.isdir(p)), None)
if bench is None:
    bench = next((p for p in glob.glob("/kaggle/input/*") if os.path.isdir(os.path.join(p, "data"))), None)
if bench is None:
    raise SystemExit("attach the dataset holding 0.1. BENCHMARK")

link = os.path.join(CHECKOUT, "0.1. BENCHMARK")
if not os.path.exists(link):
    os.symlink(bench, link)
print("corpus:", bench, "->", len(os.listdir(os.path.join(link, "data"))), "files")

## Light preparation: pdf-inspector only

`--ocr none` writes `pages_inspector.jsonl`, the merge base for pure PP-OCRv5.
It is included in the output zip so the local merge uses this exact file rather than
a second inspector run that could differ by package version.

In [ ]:
!pip install -q "pdf-inspector>=0.2.6,<0.3" pymupdf
import sys

sys.path.insert(0, CHECKOUT)
os.chdir(CHECKOUT)

import pdf_inspector
from ondemand.light_prep import run as light_prep, wants_ocr

print("pdf-inspector", getattr(pdf_inspector, "__version__", "?"))
pages_path = light_prep(ocr="none", workers=4)
print(pages_path)

In [ ]:
import json

from ondemand.bench import load_jsonl

rows = load_jsonl(pages_path)
targets = [r for r in rows if wants_ocr(r)]
print(len(targets), "of", len(rows), "pages need OCR")

## Render at 200 dpi

Same geometry as the earlier run: 200 dpi, long side capped at 4000 px, so the line
polygons stay comparable with what the debugging portal already renders.

In [ ]:
import fitz

WORK = "/kaggle/working/light_ocr"
os.makedirs(f"{WORK}/png", exist_ok=True)
MAX_SIDE = 4000
BENCH_ROOT = link

for r in targets:
    r["png"] = f"{WORK}/png/" + r["page_id"].replace("::", "__").replace("#page=", "__p") + ".png"
    if not os.path.exists(r["png"]):
        with fitz.open(os.path.join(BENCH_ROOT, r["relative_path"])) as doc:
            page = doc[r["page_index"]]
            dpi = 200
            side = max(page.rect.width, page.rect.height) * dpi / 72
            if side > MAX_SIDE:
                dpi = int(dpi * MAX_SIDE / side)
            page.get_pixmap(dpi=dpi).save(r["png"])
print("rendered", len(targets), "pages")

## PP-OCRv5

Kaggle's T4 is CUDA 12.x, so the cu126 Paddle wheel applies. On a P100 the wheel will not
load: switch the accelerator to T4 rather than letting it fall back to CPU, which would take hours.

In [ ]:
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr>=3.0,<4"

In [ ]:
import time

from paddleocr import PaddleOCR

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
started = time.perf_counter()
ocr = PaddleOCR(text_detection_model_name="PP-OCRv5_server_det",
                text_recognition_model_name="PP-OCRv5_server_rec",
                use_doc_orientation_classify=False, use_doc_unwarping=False,
                use_textline_orientation=False, device="gpu:0")
ocr.predict(targets[0]["png"])
load = time.perf_counter() - started
print(f"{gpu}: model ready in {load:.1f}s")

In [ ]:
def resume(path):
    return {json.loads(l)["unit"] for l in open(path)} if os.path.exists(path) else set()

def loop(path, produce):
    done = resume(path)
    todo = [r for r in targets if r["page_id"] not in done]
    print(f"{os.path.basename(path)}: {len(done)} done, {len(todo)} to go")
    with open(path, "a") as sink:
        for i, r in enumerate(todo, 1):
            started = time.perf_counter()
            try:
                payload, error = produce(r), None
            except Exception as exc:
                payload, error = None, f"{type(exc).__name__}: {exc}"[:200]
                print("fail", r["page_id"], error)
            sink.write(json.dumps({"unit": r["page_id"], **(payload or {}),
                                   "seconds": time.perf_counter() - started, "error": error},
                                  ensure_ascii=False) + "\n")
            sink.flush()
            if i % 50 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)}")

def text_of(r):
    return {"text": "\n".join(t for p in ocr.predict(r["png"]) for t in p["rec_texts"])}

loop(f"{WORK}/light_ocr_ppocrv5.jsonl", text_of)
json.dump({"model": "PP-OCRv5_server_det + PP-OCRv5_server_rec", "load_seconds": load, "gpu": gpu},
          open(f"{WORK}/meta_ppocrv5.json", "w"))

## Line boxes and confidences

`light_ocr.merge_pure` takes the per-page confidence from this file, and the portal re-renders the polygons.

In [ ]:
def lines_of(r):
    out = []
    for p in ocr.predict(r["png"]):
        polys = p.get("rec_polys", p.get("dt_polys", []))
        for text, score, poly in zip(p["rec_texts"], p["rec_scores"], polys):
            out.append({"text": text, "score": float(score),
                        "poly": [[float(x), float(y)] for x, y in poly]})
    return {"lines": out}

loop(f"{WORK}/light_ocr_boxes_ppocrv5.jsonl", lines_of)

## Timing and download

`seconds_per_page` is a Kaggle T4 number. Keep the GPU name beside it in any sheet.

Locally, finish with:

```
python -m ondemand merge-ocr --results light_ocr_ppocrv5.zip
```

after copying `pages_inspector.jsonl` from the zip into `data/work/ondemand_v2/<fp>/light_prep/`.

In [ ]:
import shutil
import statistics
import zipfile

shutil.copy(pages_path, f"{WORK}/pages_inspector.jsonl")
rows = [json.loads(l) for l in open(f"{WORK}/light_ocr_ppocrv5.jsonl")]
seconds = [r["seconds"] for r in rows]
print({"pages": len(rows), "failed": sum(bool(r["error"]) for r in rows),
       "empty": sum(not r.get("text", "").strip() for r in rows),
       "total_seconds": round(sum(seconds), 1),
       "seconds_per_page": round(statistics.mean(seconds), 3), "gpu": gpu})

out = "/kaggle/working/light_ocr_ppocrv5.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(WORK)):
        if name.endswith((".jsonl", ".json")):
            zf.write(f"{WORK}/{name}", name)
print("wrote", out)